<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Spatial Domain</b></h1>
</div>

This notebook executes the spatial-filtering experiments covering convolution/correlation, boundary conditions, smoothing, denoising, sharpening, derivative operators, quantitative restoration metrics, and diagnostic validation.


## Setup — Environment and Configuration

We import the libraries used throughout the laboratory.

A few helper functions are also defined so that figure saving, image display, and metric computation remain consistent.

In [ ]:
from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
import cv2

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

### Checkpoint

The roles of the main libraries are:

- **NumPy** → numerical arrays and first-principles implementations;
- **Matplotlib** → visualization;
- **Pillow** → reproducible image loading;
- **SciPy `ndimage`** → correlation, convolution, smoothing, median, derivatives;
- **OpenCV** → bilateral filtering and Scharr derivatives.

The notebook will avoid hard-coded absolute paths.

## 1. Data and Output Paths

The notebook locates the laboratory directory automatically by searching upward for both `data/` and `notebooks/`.

This makes execution robust whether VS Code starts the notebook from the repository root, the lab root, or the notebook directory.

In [ ]:
def find_lab_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the lab root. Expected a directory containing data/ and notebooks/."
    )

LAB_DIR = find_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MOON_PATH = DATA_DIR / "moon-blurred.tif"
ASCENT_PATH = DATA_DIR / "ascentB.png"

EINSTEIN_DIR = DATA_DIR / "bilateral" / "einstein"
TAJ_DIR = DATA_DIR / "bilateral" / "tajMahal"
ZEBRA_DIR = DATA_DIR / "bilateral" / "zebra"
MONARCH_DIR = DATA_DIR / "bilateral" / "monarch"

required = [
    MOON_PATH,
    ASCENT_PATH,
    EINSTEIN_DIR / "ref.png",
    EINSTEIN_DIR / "einstein_gaussian_0.png",
    EINSTEIN_DIR / "einstein_saltAndPepper.png",
]

missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing required files: {missing}"

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Required files found:", len(required))

### Data inventory

The provided dataset contains several kinds of degradations:

- **Gaussian noise** → additive fluctuations;
- **salt-and-pepper noise** → isolated extreme pixels;
- **speckle noise** → multiplicative fluctuations;
- **blurred moon image** → useful for sharpening;
- **ascent image** → useful for derivative and edge experiments.

Different degradations should not automatically be treated with the same filter.

In [ ]:
def load_gray(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"), dtype=np.uint8)

def load_rgb(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)

einstein_ref = load_gray(EINSTEIN_DIR / "ref.png")
einstein_gaussian = load_gray(EINSTEIN_DIR / "einstein_gaussian_0.png")
einstein_salt = load_gray(EINSTEIN_DIR / "einstein_saltAndPepper.png")
einstein_speckle = load_gray(EINSTEIN_DIR / "einstein_speckle_0.png")

moon = load_gray(MOON_PATH)
ascent = load_gray(ASCENT_PATH)

taj_ref = load_rgb(TAJ_DIR / "ref.jpg")
zebra_ref = load_rgb(ZEBRA_DIR / "ref.jpg")
monarch_ref = load_rgb(MONARCH_DIR / "ref.png")

for name, image in {
    "einstein_ref": einstein_ref,
    "einstein_gaussian": einstein_gaussian,
    "einstein_salt": einstein_salt,
    "einstein_speckle": einstein_speckle,
    "moon": moon,
    "ascent": ascent,
    "taj_ref": taj_ref,
    "zebra_ref": zebra_ref,
    "monarch_ref": monarch_ref,
}.items():
    print(f"{name:20s} shape={image.shape!s:16s} dtype={image.dtype} range=({image.min()}, {image.max()})")

## 2. Spatial Filtering Formulation

A **point operation** transforms one pixel using only its own value:

$$
g(x,y)=T(f(x,y))
$$

A **spatial neighborhood operation** uses nearby pixels:

$$
g(x,y)=T\left(\text{neighborhood around }(x,y)\right)
$$

This is the fundamental difference between the previous image-transformation laboratory and the present filtering laboratory.

Spatial filters can:

- suppress noise;
- blur small structures;
- preserve or destroy edges;
- emphasize rapid intensity changes;
- sharpen an image;
- estimate local derivatives.

### Neighborhoods

For a $3\times3$ neighborhood centered on pixel $(x,y)$:

$$
\begin{bmatrix}
f(x-1,y-1) & f(x,y-1) & f(x+1,y-1) \\
f(x-1,y)   & f(x,y)   & f(x+1,y) \\
f(x-1,y+1) & f(x,y+1) & f(x+1,y+1)
\end{bmatrix}
$$

In NumPy, remember that indexing is `[row, column]`, not `[x,y]`.

A filter combines values from this neighborhood according to a rule.

In [ ]:
toy = np.array(
    [
        [10, 10, 10, 10, 10],
        [10, 20, 20, 20, 10],
        [10, 20, 90, 20, 10],
        [10, 20, 20, 20, 10],
        [10, 10, 10, 10, 10],
    ],
    dtype=np.float32,
)

center_neighborhood = toy[1:4, 1:4]

print("Toy image:")
print(toy)
print("\n3x3 neighborhood around the center:")
print(center_neighborhood)

### Interpretation

The center value is `90`, but its neighbors are mostly `20`.

A smoothing filter may replace the center with a value closer to its neighborhood, while an edge/sharpening filter may emphasize the difference between the center and its surroundings.

The same neighborhood can therefore be interpreted differently depending on the filter.

## 3. Kernel Anatomy

A **kernel** (also called a mask or filter) is a small matrix of weights.

For a linear $3\times3$ filter:

$$
K=
\begin{bmatrix}
k_{-1,-1} & k_{0,-1} & k_{1,-1}\\
k_{-1,0}  & k_{0,0}  & k_{1,0}\\
k_{-1,1}  & k_{0,1}  & k_{1,1}
\end{bmatrix}
$$

Important kernel properties include:

- **size** — e.g. 3×3, 5×5, 9×9;
- **anchor / center** — location aligned with the current pixel;
- **weights** — determine how neighbors contribute;
- **sum of weights** — often controls response to constant regions;
- **symmetry** — important for smoothing and derivative behavior.

A normalized smoothing kernel usually has weights summing to 1.

In [ ]:
mean_3x3 = np.ones((3, 3), dtype=np.float32) / 9.0

sobel_x_kernel = np.array(
    [
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1],
    ],
    dtype=np.float32,
)

print("Mean kernel:")
print(mean_3x3)
print("Sum:", mean_3x3.sum())

print("\nSobel-x kernel:")
print(sobel_x_kernel)
print("Sum:", sobel_x_kernel.sum())

### Why does the kernel sum matter?

If an image region is constant with intensity $c$, then a linear filter with kernel sum 1 returns approximately the same constant:

$$
c\sum_{i,j}K(i,j)=c
$$

A derivative kernel usually has sum 0, so a perfectly constant region produces approximately zero response.

This simple test is useful when inspecting an unfamiliar kernel.

## 4. Correlation vs Convolution

Correlation and convolution both slide a kernel across an image.

The difference is the kernel orientation.

### Correlation

For correlation, the kernel is used as written.

### Convolution

For convolution, the kernel is flipped horizontally and vertically before the sliding operation.

In 2-D:

$$
K_{\mathrm{conv}}(i,j)=K(-i,-j)
$$

If a kernel is symmetric, correlation and convolution produce the same result.

If it is asymmetric, they generally differ.

In [ ]:
small = np.arange(1, 26, dtype=np.float32).reshape(5, 5)

asymmetric_kernel = np.array(
    [
        [1, 2, 0],
        [0, 0, 0],
        [0, 0, -1],
    ],
    dtype=np.float32,
)

flipped_kernel = np.flip(asymmetric_kernel, axis=(0, 1))

print("Input:")
print(small)
print("\nOriginal kernel:")
print(asymmetric_kernel)
print("\nKernel flipped for convolution:")
print(flipped_kernel)

### Manual center-pixel calculation

At the center of the 5×5 toy image, we extract a 3×3 neighborhood and compute:

$$
\text{response}=\sum \left(\text{neighborhood}\times\text{kernel}\right)
$$

This is the key operation hidden inside high-level filtering functions.

In [ ]:
patch = small[1:4, 1:4]

manual_corr = np.sum(patch * asymmetric_kernel)
manual_conv = np.sum(patch * flipped_kernel)

print("Patch:")
print(patch)
print("\nManual correlation response:", manual_corr)
print("Manual convolution response :", manual_conv)

In [ ]:
scipy_corr = ndimage.correlate(small, asymmetric_kernel, mode="constant", cval=0.0)
scipy_conv = ndimage.convolve(small, asymmetric_kernel, mode="constant", cval=0.0)

print("SciPy correlation at center:", scipy_corr[2, 2])
print("SciPy convolution at center :", scipy_conv[2, 2])

assert np.isclose(manual_corr, scipy_corr[2, 2])
assert np.isclose(manual_conv, scipy_conv[2, 2])

print("\nPASS — manual calculations agree with SciPy.")

### Common pitfall

OpenCV's `filter2D` performs **correlation**, not mathematical convolution, unless the kernel is manually flipped.

For symmetric smoothing kernels this distinction may be invisible.

For asymmetric derivative filters, it can change the sign or orientation of the response.

## 5. Direct Convolution Implementation

Implement a direct convolution routine as a numerical reference for validating library-based filtering.

The reference implementation exposes the exact numerical sequence:

```text
pad image
    ↓
extract neighborhood
    ↓
multiply by flipped kernel
    ↓
sum
    ↓
store output pixel
```

In [ ]:
def convolve2d_manual(image, kernel):
    """
    Apply a 2-D convolution manually.

    Parameters
    ----------
    image : 2-D NumPy array
        Input grayscale image.
    kernel : 2-D NumPy array
        Convolution kernel.

    Returns
    -------
    output : 2-D NumPy array
        Filtered image with the same size as the input.
    """

    image = np.asarray(image, dtype=float)
    kernel = np.asarray(kernel, dtype=float)

    kh, kw = kernel.shape

    # This implementation assumes odd-sized kernels
    assert kh % 2 == 1 and kw % 2 == 1, \
        "Kernel dimensions must be odd."

    pad_h = kh // 2
    pad_w = kw // 2

    # IMPORTANT:
    # np.pad(..., mode="symmetric") matches the border convention
    # used by scipy.ndimage.convolve(..., mode="reflect").
    padded = np.pad(
        image,
        ((pad_h, pad_h), (pad_w, pad_w)),
        mode="symmetric"
    )

    # True mathematical convolution flips the kernel.
    kernel_flipped = np.flip(kernel, axis=(0, 1))

    output = np.zeros_like(image, dtype=float)

    # Slide the kernel over every pixel
    for row in range(image.shape[0]):
        for col in range(image.shape[1]):

            neighbourhood = padded[
                row:row + kh,
                col:col + kw
            ]

            output[row, col] = np.sum(
                neighbourhood * kernel_flipped
            )

    return output

In [ ]:
# Use the same numerical precision for both implementations
toy_float64 = np.asarray(toy, dtype=np.float64)
kernel_float64 = np.asarray(mean_3x3, dtype=np.float64)

manual_result = convolve2d_manual(
    toy_float64,
    kernel_float64
)

scipy_result = ndimage.convolve(
    toy_float64,
    kernel_float64,
    mode="reflect"
)

max_difference = np.max(
    np.abs(manual_result - scipy_result)
)

print("Manual dtype:", manual_result.dtype)
print("SciPy dtype :", scipy_result.dtype)
print("Maximum absolute difference:", max_difference)

assert np.allclose(
    manual_result,
    scipy_result,
    rtol=1e-10,
    atol=1e-12
)

print("PASS — manual convolution matches scipy.ndimage.convolve.")

### Why not use the manual routine on large images?

The nested Python loops are intentionally slow.

Production implementations use optimized compiled routines, vectorization, separable filtering, FFT-based techniques, or specialized hardware.

The manual implementation is useful because it exposes the algorithmic structure.

## 6. Border Handling

At the image border, part of the neighborhood lies outside the array.

A filtering algorithm must decide what values exist beyond the image.

Common strategies include:

- `constant` → fill with a fixed value, often 0;
- `nearest` → repeat the nearest border pixel;
- `reflect` → mirror the image around the edge;
- `mirror` → a related reflection convention;
- `wrap` → continue from the opposite side.

The border rule can change the numerical result.

In [ ]:
demo = np.zeros((9, 9), dtype=np.float32)
demo[2:7, 2:7] = 200

large_mean = np.ones((5, 5), dtype=np.float32) / 25.0

border_modes = {
    "constant": ndimage.convolve(demo, large_mean, mode="constant", cval=0.0),
    "nearest": ndimage.convolve(demo, large_mean, mode="nearest"),
    "reflect": ndimage.convolve(demo, large_mean, mode="reflect"),
    "mirror": ndimage.convolve(demo, large_mean, mode="mirror"),
    "wrap": ndimage.convolve(demo, large_mean, mode="wrap"),
}

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
axes = axes.ravel()

axes[0].imshow(demo, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")

for ax, (name, result) in zip(axes[1:], border_modes.items()):
    ax.imshow(result, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_border_modes.png", bbox_inches="tight")
plt.show()

### Interpretation

Border strategies mostly affect pixels near the image boundary.

For natural-image smoothing, `reflect` is often a reasonable default because it avoids introducing a strong artificial black frame.

However, the correct choice depends on the physical meaning of the data.

## 7. Mean / Box Filtering

The $3\times3$ mean filter is:

$$
K=
\frac{1}{9}
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}
$$

Each output pixel is the arithmetic mean of its neighborhood.

The mean filter reduces local fluctuations, but it does not know whether a variation is noise or a real edge.

Therefore:

```text
larger averaging neighborhood
    → stronger smoothing
    → stronger edge/detail loss
```

In [ ]:
mean_kernel = np.ones((3, 3), dtype=np.float32) / 9.0
mean_filtered = ndimage.convolve(
    einstein_gaussian.astype(np.float32),
    mean_kernel,
    mode="reflect",
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(einstein_ref, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Reference")

axes[1].imshow(einstein_gaussian, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Gaussian-noisy")

axes[2].imshow(mean_filtered, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("3x3 mean filter")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_mean_filter.png", bbox_inches="tight")
plt.show()

### Kernel-size experiment

Kernel size controls the spatial scale of averaging.

We compare 3×3, 5×5, and 9×9 filters.

In [ ]:
mean_sizes = [3, 5, 9]
mean_results = {}

for size in mean_sizes:
    kernel = np.ones((size, size), dtype=np.float32) / (size * size)
    mean_results[size] = ndimage.convolve(
        einstein_gaussian.astype(np.float32),
        kernel,
        mode="reflect",
    )

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(einstein_gaussian, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy input")

for ax, size in zip(axes[1:], mean_sizes):
    ax.imshow(mean_results[size], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{size}x{size} mean")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_mean_kernel_sizes.png", bbox_inches="tight")
plt.show()

### Interpretation

A larger mean kernel usually reduces random noise more strongly, but it also removes more fine texture and edge contrast.

This illustrates a general image-processing trade-off:

$$
\text{noise suppression} \quad \leftrightarrow \quad \text{detail preservation}
$$

There is no universally best kernel size.

## 8. Gaussian Filtering

A Gaussian filter gives larger weight to nearby pixels and smaller weight to distant pixels.

The continuous 2-D Gaussian is:

$$
G(x,y)=
\frac{1}{2\pi\sigma^2}
\exp\left(
-\frac{x^2+y^2}{2\sigma^2}
\right)
$$

The parameter $\sigma$ controls the spatial spread:

- small $\sigma$ → weak smoothing;
- large $\sigma$ → stronger smoothing.

Unlike a box filter, the weights vary smoothly with distance.

### Build a Gaussian kernel directly from the analytical expression

A discrete Gaussian kernel is obtained by evaluating the Gaussian equation on a finite grid and normalizing the weights so that they sum to 1.

In [ ]:
def gaussian_kernel_2d(size: int, sigma: float) -> np.ndarray:
    assert size % 2 == 1, "Kernel size must be odd."
    assert sigma > 0, "sigma must be positive."

    radius = size // 2
    coords = np.arange(-radius, radius + 1, dtype=np.float64)
    x, y = np.meshgrid(coords, coords)

    kernel = np.exp(-(x**2 + y**2) / (2.0 * sigma**2))
    kernel /= kernel.sum()
    return kernel

g5 = gaussian_kernel_2d(5, 1.0)

print(np.round(g5, 4))
print("Kernel sum:", g5.sum())

### Gaussian separability

The 2-D Gaussian can be written as the product of two 1-D Gaussians:

$$
G(x,y)=G_x(x)G_y(y)
$$

This means a 2-D Gaussian filter can be implemented as:

```text
horizontal 1-D filtering
    ↓
vertical 1-D filtering
```

instead of a full 2-D convolution.

This can reduce computational cost substantially for large kernels.

In [ ]:
def gaussian_kernel_1d(size: int, sigma: float) -> np.ndarray:
    radius = size // 2
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    g = np.exp(-(x**2) / (2.0 * sigma**2))
    return g / g.sum()

g1 = gaussian_kernel_1d(5, 1.0)
outer = np.outer(g1, g1)

print("Maximum difference between 2-D kernel and outer product:")
print(np.max(np.abs(g5 - outer)))

assert np.allclose(g5, outer)

In [ ]:
gaussian_filtered = ndimage.gaussian_filter(
    einstein_gaussian.astype(np.float32),
    sigma=1.5,
    mode="reflect",
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(einstein_ref, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Reference")

axes[1].imshow(einstein_gaussian, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Gaussian-noisy")

axes[2].imshow(gaussian_filtered, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Gaussian filter, sigma=1.5")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_gaussian_filter.png", bbox_inches="tight")
plt.show()

### Effect of `sigma`

We now vary $\sigma$ while keeping the same input image.

In [ ]:
sigmas = [0.5, 1.0, 2.0, 3.0]
gaussian_results = {
    sigma: ndimage.gaussian_filter(
        einstein_gaussian.astype(np.float32),
        sigma=sigma,
        mode="reflect",
    )
    for sigma in sigmas
}

fig, axes = plt.subplots(1, 5, figsize=(17, 4))

axes[0].imshow(einstein_gaussian, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy input")

for ax, sigma in zip(axes[1:], sigmas):
    ax.imshow(gaussian_results[sigma], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"sigma={sigma}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_gaussian_sigma_sweep.png", bbox_inches="tight")
plt.show()

### Mean vs Gaussian smoothing

Both are linear low-pass smoothing filters, but their spatial weighting differs.

The mean filter gives identical weight to every pixel inside the window.

The Gaussian filter emphasizes nearby pixels and reduces the influence of pixels farther from the center.

In [ ]:
mean_5 = ndimage.uniform_filter(
    einstein_gaussian.astype(np.float32),
    size=5,
    mode="reflect",
)
gauss_approx = ndimage.gaussian_filter(
    einstein_gaussian.astype(np.float32),
    sigma=1.0,
    mode="reflect",
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [einstein_gaussian, mean_5, gauss_approx],
    ["Noisy", "5x5 mean", "Gaussian sigma=1"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_mean_vs_gaussian.png", bbox_inches="tight")
plt.show()

## 9. Why Noise Type Matters

The same filter should not be selected blindly for every degradation.

The provided Einstein images let us compare three important noise types:

- Gaussian;
- salt-and-pepper;
- speckle.

Their visual structure is different, so their preferred filters can also differ.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, img, title in zip(
    axes,
    [einstein_ref, einstein_gaussian, einstein_salt, einstein_speckle],
    ["Reference", "Gaussian noise", "Salt-and-pepper", "Speckle noise"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_noise_types.png", bbox_inches="tight")
plt.show()

### Key idea

A linear average works by combining neighboring intensities.

This is effective when noise is spread as moderate fluctuations.

Impulse noise is different: one corrupted pixel may jump directly to 0 or 255. Such an extreme value can strongly distort an arithmetic average.

This motivates the median filter.

## 10. Median Filtering

The median filter is **nonlinear**.

For each neighborhood:

1. collect all pixel values;
2. sort them;
3. select the middle value;
4. assign that median to the output pixel.

Example neighborhood values:

```text
[20, 21, 20,
 22, 255, 19,
 20, 21, 20]
```

The value `255` is an impulse outlier.

The median remains near the normal neighborhood values instead of being pulled strongly upward.

In [ ]:
values = np.array([20, 21, 20, 22, 255, 19, 20, 21, 20])

print("Values:", values)
print("Sorted:", np.sort(values))
print("Mean  :", values.mean())
print("Median:", np.median(values))

In [ ]:
median_filtered = ndimage.median_filter(
    einstein_salt,
    size=3,
    mode="reflect",
)

mean_on_salt = ndimage.uniform_filter(
    einstein_salt.astype(np.float32),
    size=3,
    mode="reflect",
)

gaussian_on_salt = ndimage.gaussian_filter(
    einstein_salt.astype(np.float32),
    sigma=1.0,
    mode="reflect",
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, img, title in zip(
    axes,
    [einstein_salt, mean_on_salt, gaussian_on_salt, median_filtered],
    ["Salt-and-pepper", "Mean", "Gaussian", "Median"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_median_filter.png", bbox_inches="tight")
plt.show()

### Median-window experiment

A larger median window can remove more impulse noise, but it can also remove thin structures and fine details.

In [ ]:
median_sizes = [3, 5, 7]
median_results = {
    size: ndimage.median_filter(einstein_salt, size=size, mode="reflect")
    for size in median_sizes
}

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(einstein_salt, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy")

for ax, size in zip(axes[1:], median_sizes):
    ax.imshow(median_results[size], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{size}x{size} median")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_median_sizes.png", bbox_inches="tight")
plt.show()

### Common pitfall

Median filtering is not convolution.

There is no fixed set of weights whose weighted sum produces the median.

Therefore, median filtering is an **order-statistic nonlinear filter**.

## 11. Bilateral Filtering

A bilateral filter smooths pixels using two notions of similarity:

1. **spatial similarity** — nearby pixels matter more;
2. **intensity similarity** — pixels with similar intensity matter more.

A simplified bilateral weight between a center pixel $p$ and a neighbor $q$ is:

$$
w(p,q)
=
\exp\left(
-\frac{\|p-q\|^2}{2\sigma_s^2}
\right)
\exp\left(
-\frac{|I_p-I_q|^2}{2\sigma_r^2}
\right)
$$

where:

- $\sigma_s$ controls spatial distance;
- $\sigma_r$ controls intensity/range similarity.

Because pixels across a strong edge have very different intensities, their contribution can be reduced.

### Parameter interpretation

In OpenCV:

- `d` → neighborhood diameter;
- `sigmaSpace` → how far spatial influence extends;
- `sigmaColor` → how tolerant the filter is to intensity/color differences.

Typical behavior:

```text
larger sigmaSpace
    → broader spatial smoothing

larger sigmaColor
    → more cross-edge mixing
    → less edge preservation
```

In [ ]:
bilateral = cv2.bilateralFilter(
    einstein_gaussian,
    d=9,
    sigmaColor=45,
    sigmaSpace=45,
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, img, title in zip(
    axes,
    [einstein_ref, einstein_gaussian, gaussian_filtered, bilateral],
    ["Reference", "Noisy", "Gaussian filter", "Bilateral filter"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_bilateral_filter.png", bbox_inches="tight")
plt.show()

### Bilateral parameter sweep

We vary `sigmaColor` while keeping the spatial parameter fixed.

In [ ]:
sigma_colors = [10, 30, 60, 120]

bilateral_results = {
    sc: cv2.bilateralFilter(
        einstein_gaussian,
        d=9,
        sigmaColor=sc,
        sigmaSpace=45,
    )
    for sc in sigma_colors
}

fig, axes = plt.subplots(1, 5, figsize=(17, 4))

axes[0].imshow(einstein_gaussian, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy")

for ax, sc in zip(axes[1:], sigma_colors):
    ax.imshow(bilateral_results[sc], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"sigmaColor={sc}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_bilateral_parameter_sweep.png", bbox_inches="tight")
plt.show()

### Interpretation

When `sigmaColor` is small, pixels with noticeably different intensity contribute little, so edges are protected strongly.

When `sigmaColor` becomes very large, intensity differences matter less and the filter behaves more like ordinary spatial smoothing.

Bilateral filtering is useful when edge preservation matters, but it is computationally more expensive than basic linear smoothing.

## 12. Quantitative Denoising Metrics

Visual inspection is essential, but quantitative metrics help compare outputs reproducibly.

For reference image $R$ and test image $T$:

### MAE

$$
\mathrm{MAE}
=
\frac{1}{N}\sum |R-T|
$$

### MSE

$$
\mathrm{MSE}
=
\frac{1}{N}\sum (R-T)^2
$$

### RMSE

$$
\mathrm{RMSE}=\sqrt{\mathrm{MSE}}
$$

### PSNR

For 8-bit images:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

Higher PSNR normally means lower pixel-wise error.

In [ ]:
def mae(reference, test):
    ref = reference.astype(np.float64)
    tst = np.asarray(test, dtype=np.float64)
    return float(np.mean(np.abs(ref - tst)))

def mse(reference, test):
    ref = reference.astype(np.float64)
    tst = np.asarray(test, dtype=np.float64)
    return float(np.mean((ref - tst) ** 2))

def rmse(reference, test):
    return math.sqrt(mse(reference, test))

def psnr(reference, test, peak=255.0):
    value = mse(reference, test)
    if value == 0:
        return float("inf")
    return 10.0 * math.log10((peak ** 2) / value)

def metric_row(name, reference, test):
    return (
        name,
        mae(reference, test),
        mse(reference, test),
        rmse(reference, test),
        psnr(reference, test),
    )

### Compare filters on Gaussian noise

We compare the noisy image and several filtered outputs against the clean Einstein reference.

In [ ]:
gaussian_candidates = [
    metric_row("Noisy", einstein_ref, einstein_gaussian),
    metric_row("Mean 3x3", einstein_ref, mean_filtered),
    metric_row("Gaussian sigma=1.5", einstein_ref, gaussian_filtered),
    metric_row("Bilateral", einstein_ref, bilateral),
]

print(f"{'Method':24s} {'MAE':>10s} {'MSE':>12s} {'RMSE':>10s} {'PSNR(dB)':>10s}")
print("-" * 72)

for name, a, m, r, p in gaussian_candidates:
    print(f"{name:24s} {a:10.3f} {m:12.3f} {r:10.3f} {p:10.3f}")

### Compare filters on salt-and-pepper noise

This experiment demonstrates why median filtering is often preferred for impulse noise.

In [ ]:
salt_candidates = [
    metric_row("Noisy", einstein_ref, einstein_salt),
    metric_row("Mean 3x3", einstein_ref, mean_on_salt),
    metric_row("Gaussian sigma=1", einstein_ref, gaussian_on_salt),
    metric_row("Median 3x3", einstein_ref, median_filtered),
]

print(f"{'Method':24s} {'MAE':>10s} {'MSE':>12s} {'RMSE':>10s} {'PSNR(dB)':>10s}")
print("-" * 72)

for name, a, m, r, p in salt_candidates:
    print(f"{name:24s} {a:10.3f} {m:12.3f} {r:10.3f} {p:10.3f}")

### Important limitation of PSNR

A high PSNR does not guarantee that an image looks perceptually best.

A very smooth image may obtain a reasonable pixel-wise score while losing important edges or texture.

Therefore the recommended evaluation is:

```text
numerical metrics
    +
visual inspection
    +
task-specific requirements
```

## 13. Edge Preservation as a Secondary Check

One simple way to inspect structural preservation is to compare gradient magnitude.

This is not a universal perceptual-quality metric, but it gives useful intuition about whether smoothing has weakened edges.

In [ ]:
def sobel_magnitude(image):
    arr = np.asarray(image, dtype=np.float32)
    gx = ndimage.sobel(arr, axis=1, mode="reflect")
    gy = ndimage.sobel(arr, axis=0, mode="reflect")
    return np.hypot(gx, gy)

edge_ref = sobel_magnitude(einstein_ref)
edge_mean = sobel_magnitude(mean_filtered)
edge_gauss = sobel_magnitude(gaussian_filtered)
edge_bilat = sobel_magnitude(bilateral)

print("Mean gradient magnitude")
print(f"Reference : {edge_ref.mean():.3f}")
print(f"Mean      : {edge_mean.mean():.3f}")
print(f"Gaussian  : {edge_gauss.mean():.3f}")
print(f"Bilateral : {edge_bilat.mean():.3f}")

## 14. Sharpening with the Laplacian

Smoothing suppresses high local variation.

Sharpening does the opposite: it emphasizes rapid intensity changes.

The continuous Laplacian is:

$$
\nabla^2 f
=
\frac{\partial^2 f}{\partial x^2}
+
\frac{\partial^2 f}{\partial y^2}
$$

A common discrete 4-neighbor Laplacian kernel is:

$$
\begin{bmatrix}
0&1&0\\
1&-4&1\\
0&1&0
\end{bmatrix}
$$

Because Laplacian sign conventions differ between implementations, the sharpening formula must be checked carefully.

In [ ]:
laplacian_kernel = np.array(
    [
        [0, 1, 0],
        [1, -4, 1],
        [0, 1, 0],
    ],
    dtype=np.float32,
)

print(laplacian_kernel)
print("Kernel sum:", laplacian_kernel.sum())

In [ ]:
moon_float = moon.astype(np.float32)

laplacian = ndimage.convolve(
    moon_float,
    laplacian_kernel,
    mode="reflect",
)

lap_sharp = np.clip(
    moon_float - laplacian,
    0,
    255,
).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(moon, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Blurred moon")

axes[1].imshow(laplacian, cmap="gray")
axes[1].set_title("Laplacian response")

axes[2].imshow(lap_sharp, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Laplacian sharpened")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_laplacian_sharpening.png", bbox_inches="tight")
plt.show()

### Common pitfall — sign convention

Some libraries define the Laplacian with the opposite sign.

Therefore, do not memorize only:

```python
sharp = image - laplacian
```

or only:

```python
sharp = image + laplacian
```

Instead:

1. inspect the kernel/sign convention;
2. test on a simple edge;
3. verify visually and numerically.

## 15. Unsharp Masking and High-Boost Filtering

Unsharp masking first creates a blurred version of the image.

The detail mask is:

$$
m=f-f_{\mathrm{blur}}
$$

Then the sharpened image is:

$$
g=f+k\,m
$$

where $k$ controls sharpening strength.

- $k=1$ → classical unsharp masking;
- $k>1$ → stronger high-boost sharpening.

In [ ]:
blurred = ndimage.gaussian_filter(moon_float, sigma=2.0, mode="reflect")
detail_mask = moon_float - blurred

unsharp = np.clip(
    moon_float + 1.0 * detail_mask,
    0,
    255,
).astype(np.uint8)

high_boost = np.clip(
    moon_float + 1.8 * detail_mask,
    0,
    255,
).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, img, title in zip(
    axes,
    [moon, blurred, unsharp, high_boost],
    ["Input", "Gaussian blur", "Unsharp k=1", "High-boost k=1.8"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "13_unsharp_highboost.png", bbox_inches="tight")
plt.show()

### Sharpening trade-off

Sharpening can improve local contrast, but excessive sharpening may amplify:

- noise;
- ringing;
- halos;
- quantization artifacts.

Sharpening is therefore not equivalent to recovering lost information.

## 16. First Derivatives and Image Gradients

Edges correspond to rapid intensity variation.

The image gradient contains horizontal and vertical derivatives:

$$
\nabla f=
\begin{bmatrix}
G_x\\
G_y
\end{bmatrix}
$$

The gradient magnitude is:

$$
|\nabla f|
=
\sqrt{G_x^2+G_y^2}
$$

The gradient orientation is:

$$
\theta
=
\operatorname{atan2}(G_y,G_x)
$$

The Sobel operator combines differentiation with a small amount of local smoothing.

In [ ]:
gx = ndimage.sobel(ascent.astype(np.float32), axis=1, mode="reflect")
gy = ndimage.sobel(ascent.astype(np.float32), axis=0, mode="reflect")

magnitude = np.hypot(gx, gy)
orientation = np.arctan2(gy, gx)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")

axes[1].imshow(np.abs(gx), cmap="gray")
axes[1].set_title("|Gx|")

axes[2].imshow(np.abs(gy), cmap="gray")
axes[2].set_title("|Gy|")

axes[3].imshow(magnitude, cmap="gray")
axes[3].set_title("Gradient magnitude")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "14_sobel_gradients.png", bbox_inches="tight")
plt.show()

### Interpreting direction

A strong vertical edge corresponds to a large change while moving horizontally, so it produces a strong $G_x$ response.

A strong horizontal edge corresponds to a large change while moving vertically, so it produces a strong $G_y$ response.

This is a frequent source of confusion.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(orientation, cmap="twilight", vmin=-np.pi, vmax=np.pi)
ax.set_title("Gradient orientation (radians)")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "15_gradient_orientation.png", bbox_inches="tight")
plt.show()

## 17. Sobel vs Prewitt vs Scharr

Several derivative operators approximate image gradients.

### Prewitt

Uses simple derivative and smoothing weights.

### Sobel

Gives larger weight to the center row/column and is extremely common.

### Scharr

Uses coefficients designed to improve rotational symmetry for a 3×3 derivative operator.

No operator is universally best for every problem.

In [ ]:
ascent_f = ascent.astype(np.float32)

prewitt_x = ndimage.prewitt(ascent_f, axis=1, mode="reflect")
prewitt_y = ndimage.prewitt(ascent_f, axis=0, mode="reflect")
prewitt_mag = np.hypot(prewitt_x, prewitt_y)

sobel_x = ndimage.sobel(ascent_f, axis=1, mode="reflect")
sobel_y = ndimage.sobel(ascent_f, axis=0, mode="reflect")
sobel_mag = np.hypot(sobel_x, sobel_y)

scharr_x = cv2.Scharr(ascent_f, cv2.CV_32F, 1, 0)
scharr_y = cv2.Scharr(ascent_f, cv2.CV_32F, 0, 1)
scharr_mag = np.hypot(scharr_x, scharr_y)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, img, title in zip(
    axes,
    [prewitt_mag, sobel_mag, scharr_mag],
    ["Prewitt", "Sobel", "Scharr"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "16_derivative_operators.png", bbox_inches="tight")
plt.show()

## 18. Border Effects on a Real Image

Large kernels make border behavior easier to see.

We compare several padding modes using a 15×15 averaging filter.

In [ ]:
kernel_15 = np.ones((15, 15), dtype=np.float32) / (15 * 15)
ascent_f = ascent.astype(np.float32)

padding_results = {
    "constant": ndimage.convolve(ascent_f, kernel_15, mode="constant", cval=0.0),
    "nearest": ndimage.convolve(ascent_f, kernel_15, mode="nearest"),
    "reflect": ndimage.convolve(ascent_f, kernel_15, mode="reflect"),
    "wrap": ndimage.convolve(ascent_f, kernel_15, mode="wrap"),
}

fig, axes = plt.subplots(1, 5, figsize=(17, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")

for ax, (name, img) in zip(axes[1:], padding_results.items()):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "17_padding_real_image.png", bbox_inches="tight")
plt.show()

### Interpretation

`constant` padding with zero can create artificial dark influence near the border.

`wrap` assumes the opposite edge continues the image, which is often physically unrealistic for photographs.

`reflect` and `nearest` frequently produce less visually intrusive boundaries, but the best choice depends on the application.

## 19. Filtering RGB Images

A color image has shape:

```text
(H, W, 3)
```

A spatial filter should normally act over the two spatial dimensions while preserving the channel dimension.

For a Gaussian filter in SciPy, this can be expressed using:

```python
sigma=(sigma_y, sigma_x, 0)
```

The zero prevents smoothing across the channel axis.

In [ ]:
taj_gaussian = ndimage.gaussian_filter(
    taj_ref.astype(np.float32),
    sigma=(1.5, 1.5, 0),
    mode="reflect",
)

taj_bilateral = cv2.bilateralFilter(
    taj_ref,
    d=9,
    sigmaColor=45,
    sigmaSpace=45,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, img, title in zip(
    axes,
    [taj_ref, np.clip(taj_gaussian, 0, 255).astype(np.uint8), taj_bilateral],
    ["RGB reference", "Gaussian RGB", "Bilateral RGB"],
):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "18_color_filtering.png", bbox_inches="tight")
plt.show()

### Common pitfall — filtering across channels

Treating an RGB image as an ordinary 3-D volume can accidentally mix values between the red, green, and blue channels.

Spatial filtering and channel mixing are different operations.

Always check which dimensions the filtering function processes.

## 20. Numerical Safety

Filtering often produces floating-point values or signed derivative responses.

Important rules:

1. Convert to floating point before operations that may become negative or exceed 255.
2. Do not immediately cast derivative images to `uint8`.
3. Clip only when producing a display/storage image that requires a bounded range.
4. Keep signed responses when the sign contains information.
5. Normalize kernels deliberately rather than automatically.

In [ ]:
example_uint8 = np.array([0, 10, 250, 255], dtype=np.uint8)

unsafe = example_uint8 + np.uint8(20)
safe = example_uint8.astype(np.float32) + 20

print("Original:", example_uint8)
print("Unsafe uint8 + 20:", unsafe)
print("Safe float + 20 :", safe)
print("Clipped result  :", np.clip(safe, 0, 255).astype(np.uint8))

### Why derivative outputs need signed values

A derivative can be positive or negative depending on transition direction.

For example:

```text
dark → bright  : one sign
bright → dark  : opposite sign
```

If a signed derivative is converted directly to `uint8`, negative values can be lost or wrapped, destroying information.

## 21. Choosing a Filter

A useful first decision table is:

| Situation | Reasonable first choice | Why |
|---|---|---|
| Mild additive Gaussian noise | Gaussian filter | smooth weighted averaging |
| Random impulse/salt-and-pepper noise | Median filter | rejects isolated extreme values |
| Noise with important edges | Bilateral filter | weights both distance and intensity similarity |
| General simple smoothing | Mean or Gaussian | simple neighborhood averaging |
| Blur requiring local contrast enhancement | Unsharp mask / Laplacian | emphasizes high local variation |
| Edge/gradient estimation | Sobel / Scharr / Prewitt | approximates spatial derivatives |

This table is a starting point, not a substitute for validation.

## 22. Standard Spatial-Filtering Workflow

A robust workflow is:

```text
1. Inspect image
    ↓
2. Identify degradation or objective
    ↓
3. Check dtype and intensity range
    ↓
4. Choose filter family
    ↓
5. Choose border handling
    ↓
6. Start with conservative parameters
    ↓
7. Apply filter in floating point when needed
    ↓
8. Compare visually
    ↓
9. Compare numerically if reference exists
    ↓
10. Check edge/detail preservation
    ↓
11. Tune parameters
    ↓
12. Validate and save result
```

## 23. Validation Checks

The following assertions verify key mathematical and implementation assumptions.

In [ ]:
# Shapes must be preserved for same-size filtering.
assert mean_filtered.shape == einstein_ref.shape
assert gaussian_filtered.shape == einstein_ref.shape
assert median_filtered.shape == einstein_ref.shape
assert bilateral.shape == einstein_ref.shape

# Mean smoothing kernel must preserve a constant field.
assert np.isclose(mean_kernel.sum(), 1.0)

# Gaussian kernel must be normalized.
assert np.isclose(g5.sum(), 1.0)

# Derivative kernels should have approximately zero DC response.
assert np.isclose(sobel_x_kernel.sum(), 0.0)
assert np.isclose(laplacian_kernel.sum(), 0.0)

# Manual convolution must agree with SciPy on the toy example.
assert np.allclose(manual_result, scipy_result)

# Gradient magnitude must be non-negative.
assert np.all(magnitude >= 0)

# RGB filtering must preserve the three-channel image shape.
assert taj_gaussian.shape == taj_ref.shape
assert taj_bilateral.shape == taj_ref.shape

# Valid storage outputs must remain in the 8-bit range.
assert lap_sharp.min() >= 0 and lap_sharp.max() <= 255
assert unsharp.min() >= 0 and unsharp.max() <= 255

print("All spatial-filtering validation checks passed.")

## Final Result Summary

The notebook implements and validates the complete spatial-filtering workflow, including convolution/correlation, denoising, sharpening, gradients, border handling, RGB processing, and numerical safety. Generated figures are stored under `../outputs/figures/`.
